In [13]:
"""
DigiCow Farmer Training Adoption Challenge — Solution v2
=========================================================
Améliorations vs v1:
  1. LightGBM + XGBoost fortement recommandés (pip install lightgbm xgboost)
  2. Features enrichies : ratios, interactions 2nd ordre, rank features
  3. Calibration isotonique (meilleure que Platt pour log loss)
  4. Meta-learner XGBoost au lieu de LogReg
  5. Pseudo-labeling haute confiance
  6. N_splits=10 pour OOF plus stable

UTILISATION:
  pip install lightgbm xgboost catboost optuna
  python solution_v2.py
"""

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV

USE_LGBM     = False
USE_XGB      = False
USE_CATBOOST = False

try:
    import lightgbm as lgb
    USE_LGBM = True
    print("✓ LightGBM disponible")
except ImportError:
    print("✗ LightGBM non disponible")

try:
    import xgboost as xgb
    USE_XGB = True
    print("✓ XGBoost disponible")
except ImportError:
    print("✗ XGBoost non disponible")

try:
    from catboost import CatBoostClassifier
    USE_CATBOOST = True
    print("✓ CatBoost disponible")
except ImportError:
    print("✗ CatBoost non disponible")


# ============================================================
# PARAMÈTRES GLOBAUX
# ============================================================
N_SPLITS      = 10   # Plus de folds = OOF plus stable
PSEUDO_THR_HI = 0.85  # Seuil confiance haute pour pseudo-labeling (positifs)
PSEUDO_THR_LO = 0.02  # Seuil confiance basse (négatifs)
USE_PSEUDO    = True   # Activer pseudo-labeling


# ============================================================
# 1. PARSING
# ============================================================

def parse_topics(topics_str):
    try:
        raw = ast.literal_eval(str(topics_str))
        topics = []
        for item in raw:
            if isinstance(item, list):
                topics.extend([t for t in item if isinstance(t, str)])
            elif isinstance(item, str):
                topics.append(item)
        return [t.strip().lower() for t in topics if t.strip()]
    except Exception:
        return []

def parse_trainers(trainer_str):
    try:
        raw = ast.literal_eval(str(trainer_str))
        if isinstance(raw, list):
            return [str(t) for t in raw]
        return [str(raw)]
    except Exception:
        return [str(trainer_str)]


# ============================================================
# 2. STATISTIQUES PRIOR
# ============================================================

def build_prior_stats(prior):
    print("  Calcul des stats Prior...")
    prior = prior.copy()
    prior['training_day'] = pd.to_datetime(prior['training_day'])
    prior['month']     = prior['training_day'].dt.month
    prior['dayofweek'] = prior['training_day'].dt.dayofweek
    prior['quarter']   = prior['training_day'].dt.quarter
    prior['topics_parsed'] = prior['topics_list'].apply(parse_topics)
    prior['trainer_first'] = prior['trainer'].apply(
        lambda x: parse_trainers(x)[0] if parse_trainers(x) else 'unknown'
    )

    TARGETS_3 = ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days']
    stats = {}
    global_rates = {t: prior[t].mean() for t in TARGETS_3}
    stats['global_rates'] = global_rates

    # Target encoding avec Bayesian smoothing
    SMOOTH_K = 50
    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first',
                'gender', 'age', 'registration', 'month', 'dayofweek', 'quarter',
                'belong_to_cooperative', 'has_topic_trained_on']:
        if col not in prior.columns:
            continue
        for target in TARGETS_3:
            gr = global_rates[target]
            agg = prior.groupby(col)[target].agg(['sum', 'count'])
            agg['rate'] = (agg['sum'] + SMOOTH_K * gr) / (agg['count'] + SMOOTH_K)
            stats[f'{col}_{target}_rate'] = agg['rate'].to_dict()
            # Variance (incertitude)
            agg['var'] = (agg['rate'] * (1 - agg['rate'])) / (agg['count'] + 1)
            stats[f'{col}_{target}_var'] = agg['var'].to_dict()

    # Target encoding par sujet
    rows = []
    for _, row in prior.iterrows():
        for topic in row['topics_parsed']:
            rows.append({
                'topic': topic,
                'adopted_within_07_days':  row['adopted_within_07_days'],
                'adopted_within_90_days':  row['adopted_within_90_days'],
                'adopted_within_120_days': row['adopted_within_120_days'],
            })
    if rows:
        topic_df = pd.DataFrame(rows)
        smooth_k = 30
        for target in TARGETS_3:
            gr = global_rates[target]
            agg = topic_df.groupby('topic')[target].agg(['sum', 'count'])
            agg['rate'] = (agg['sum'] + smooth_k * gr) / (agg['count'] + smooth_k)
            stats[f'topic_{target}_rate'] = agg['rate'].to_dict()
            stats[f'topic_{target}_count'] = agg['count'].to_dict()

    # Encodages de volume
    stats['farmer_prior_count']   = prior['farmer_name'].value_counts().to_dict()
    stats['group_size']           = prior['group_name'].value_counts().to_dict()
    stats['trainer_event_count']  = prior['trainer_first'].value_counts().to_dict()
    stats['county_event_count']   = prior['county'].value_counts().to_dict()
    stats['ward_event_count']     = prior['ward'].value_counts().to_dict()
    stats['subcounty_event_count']= prior['subcounty'].value_counts().to_dict()

    # Taux par combinaison (interactions)
    SMOOTH_K2 = 20
    for (c1, c2) in [('county', 'gender'), ('county', 'age'), ('ward', 'belong_to_cooperative'),
                     ('group_name', 'gender'), ('county', 'registration')]:
        if c1 not in prior.columns or c2 not in prior.columns:
            continue
        prior[f'{c1}_{c2}'] = prior[c1].astype(str) + '_' + prior[c2].astype(str)
        for target in TARGETS_3:
            gr = global_rates[target]
            agg = prior.groupby(f'{c1}_{c2}')[target].agg(['sum', 'count'])
            agg['rate'] = (agg['sum'] + SMOOTH_K2 * gr) / (agg['count'] + SMOOTH_K2)
            stats[f'{c1}_{c2}_{target}_rate'] = agg['rate'].to_dict()

    return stats


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

TOPIC_KEYWORDS = {
    'dairy':   ['dairy', 'milk', 'cow', 'calf', 'lactating', 'calving', 'unga', 'transition'],
    'poultry': ['poultry', 'chicken', 'layer', 'broiler', 'kienyeji', 'egg', 'hatch'],
    'crop':    ['maize', 'bean', 'seed', 'fertilizer', 'weed', 'pest', 'crop', 'asili'],
    'health':  ['health', 'vaccine', 'vaccin', 'biosecurity', 'disease', 'ppe', 'hygiene'],
    'digital': ['app', 'ndume', 'record', 'digital'],
    'feeding': ['feeding', 'feed', 'nutrition', 'tyari'],
    'housing': ['housing', 'house', 'shed', 'pen', 'cage'],
    'finance': ['finance', 'saving', 'loan', 'credit', 'insurance'],
}

def feature_engineer(df, prior_stats, is_train=True):
    df = df.copy()
    gr = prior_stats['global_rates']
    TARGETS_3 = ['adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days']
    SHORT_MAP  = [('07', 'adopted_within_07_days'),
                  ('90', 'adopted_within_90_days'),
                  ('120', 'adopted_within_120_days')]

    # ---- Dates ----
    df['training_day'] = pd.to_datetime(df['training_day'])
    df['month']        = df['training_day'].dt.month
    df['dayofweek']    = df['training_day'].dt.dayofweek
    df['week']         = df['training_day'].dt.isocalendar().week.astype(int)
    df['quarter']      = df['training_day'].dt.quarter
    df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
    df['day_of_month'] = df['training_day'].dt.day
    df['is_month_end'] = (df['day_of_month'] >= 25).astype(int)
    df['is_month_start'] = (df['day_of_month'] <= 5).astype(int)

    # ---- Encodages simples ----
    age_map = {'Below 18': 0, '18-25': 1, '26-35': 2, 'Above 35': 3}
    df['age_enc'] = df['age'].map(age_map).fillna(2).astype(int)
    df['gender_enc'] = (df['gender'] == 'Female').astype(int)
    reg_map = {'Ussd': 1, 'Manual': 0, 'App': 2}
    df['registration_enc'] = df['registration'].map(reg_map).fillna(0).astype(int)

    # ---- Topics ----
    df['topics_parsed']  = df['topics_list'].apply(parse_topics)
    df['n_topics']       = df['topics_parsed'].apply(len)
    df['n_unique_topics']= df['topics_parsed'].apply(lambda x: len(set(x)))
    df['topics_ratio']   = df['n_unique_topics'] / (df['n_topics'] + 1)

    for cat, keywords in TOPIC_KEYWORDS.items():
        df[f'n_{cat}_topics'] = df['topics_parsed'].apply(
            lambda ts: sum(any(kw in t for kw in keywords) for t in ts)
        )
    df['n_category_types'] = sum(
        (df[f'n_{cat}_topics'] > 0).astype(int) for cat in TOPIC_KEYWORDS
    )
    df['has_digital_topic'] = (df['n_digital_topics'] > 0).astype(int)
    df['has_dairy_topic']   = (df['n_dairy_topics'] > 0).astype(int)

    # Target encoding par topic (avg/max/min + std)
    for short, target in SHORT_MAP:
        rate_dict  = prior_stats.get(f'topic_{target}_rate', {})
        count_dict = prior_stats.get(f'topic_{target}_count', {})
        default = gr[target]
        df[f'topic_avg_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.mean([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_max_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.max([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_min_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.min([rate_dict.get(t, default) for t in ts]) if ts else default
        )
        df[f'topic_std_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.std([rate_dict.get(t, default) for t in ts]) if len(ts) > 1 else 0
        )
        df[f'topic_sum_{short}_count'] = df['topics_parsed'].apply(
            lambda ts: sum(count_dict.get(t, 0) for t in ts)
        )

    # ---- Trainers ----
    df['trainers_parsed']     = df['trainer'].apply(parse_trainers)
    df['n_trainers']          = df['trainers_parsed'].apply(len)
    df['trainer_first']       = df['trainers_parsed'].apply(lambda x: x[0] if x else 'unknown')
    df['trainer_event_count'] = df['trainer_first'].map(
        prior_stats.get('trainer_event_count', {})).fillna(1)
    df['trainer_event_log']   = np.log1p(df['trainer_event_count'])

    # ---- Target encoding catégorielle ----
    cat_cols = [
        ('county', 'county'), ('subcounty', 'subcounty'), ('ward', 'ward'),
        ('group_name', 'group_name'), ('trainer_first', 'trainer_first'),
        ('gender', 'gender'), ('age', 'age'), ('registration', 'registration'),
        ('month', 'month'), ('dayofweek', 'dayofweek'), ('quarter', 'quarter'),
        ('belong_to_cooperative', 'belong_to_cooperative'),
        ('has_topic_trained_on', 'has_topic_trained_on'),
    ]
    for col, col_key in cat_cols:
        if col not in df.columns:
            continue
        for short, target in SHORT_MAP:
            rate_dict = prior_stats.get(f'{col_key}_{target}_rate', {})
            var_dict  = prior_stats.get(f'{col_key}_{target}_var', {})
            default   = gr[target]
            df[f'{col_key}_{short}_rate'] = df[col].map(rate_dict).fillna(default)
            df[f'{col_key}_{short}_var']  = df[col].map(var_dict).fillna(0)

    # ---- Target encoding interactions ----
    for (c1, c2) in [('county', 'gender'), ('county', 'age'), ('ward', 'belong_to_cooperative'),
                     ('group_name', 'gender'), ('county', 'registration')]:
        if c1 not in df.columns or c2 not in df.columns:
            continue
        df[f'{c1}_{c2}'] = df[c1].astype(str) + '_' + df[c2].astype(str)
        for short, target in SHORT_MAP:
            rate_dict = prior_stats.get(f'{c1}_{c2}_{target}_rate', {})
            default   = gr[target]
            df[f'{c1}_{c2}_{short}_rate'] = df[f'{c1}_{c2}'].map(rate_dict).fillna(default)

    # ---- Volumes & comptages ----
    df['farmer_prior_count']   = df['farmer_name'].map(
        prior_stats.get('farmer_prior_count', {})).fillna(0)
    df['farmer_experienced']   = (df['farmer_prior_count'] > 1).astype(int)
    df['group_size']           = df['group_name'].map(
        prior_stats.get('group_size', {})).fillna(1)
    df['group_size_log']       = np.log1p(df['group_size'])
    df['farmer_prior_log']     = np.log1p(df['farmer_prior_count'])
    df['county_event_count']   = df['county'].map(
        prior_stats.get('county_event_count', {})).fillna(1)
    df['county_event_log']     = np.log1p(df['county_event_count'])
    df['ward_event_count']     = df['ward'].map(
        prior_stats.get('ward_event_count', {})).fillna(1)
    df['ward_event_log']       = np.log1p(df['ward_event_count'])

    # ---- Interactions 2ème ordre clés ----
    for short in ['07', '90', '120']:
        df[f'coop_county_{short}']    = df['belong_to_cooperative'] * df[f'county_{short}_rate']
        df[f'gender_county_{short}']  = df['gender_enc'] * df[f'county_{short}_rate']
        df[f'topic_county_{short}']   = df[f'topic_avg_{short}_rate'] * df[f'county_{short}_rate']
        df[f'trainer_county_{short}'] = df[f'trainer_first_{short}_rate'] * df[f'county_{short}_rate']
        df[f'age_county_{short}']     = df['age_enc'] * df[f'county_{short}_rate']
        df[f'reg_county_{short}']     = df['registration_enc'] * df[f'county_{short}_rate']
        df[f'topic_coop_{short}']     = df[f'topic_avg_{short}_rate'] * df['belong_to_cooperative']
        df[f'ward_coop_{short}']      = df[f'ward_{short}_rate'] * df['belong_to_cooperative']
        # Ratio topic/county
        df[f'topic_vs_county_{short}']= (
            df[f'topic_avg_{short}_rate'] / (df[f'county_{short}_rate'] + 1e-6)
        )
        # Score composite (signal fort)
        df[f'composite_{short}'] = (
            0.3 * df[f'county_{short}_rate'] +
            0.3 * df[f'topic_avg_{short}_rate'] +
            0.2 * df[f'ward_{short}_rate'] +
            0.1 * df[f'group_name_{short}_rate'] +
            0.1 * df[f'trainer_first_{short}_rate']
        )

    df['topics_coop']      = df['n_topics'] * df['belong_to_cooperative']
    df['age_coop']         = df['age_enc'] * df['belong_to_cooperative']
    df['experienced_coop'] = df['farmer_experienced'] * df['belong_to_cooperative']
    df['digital_coop']     = df['has_digital_topic'] * df['belong_to_cooperative']
    df['n_topics_x_reg']   = df['n_topics'] * df['registration_enc']

    # ---- Rank features ----
    short_to_target = {
        '07':  'adopted_within_07_days',
        '90':  'adopted_within_90_days',
        '120': 'adopted_within_120_days',
    }
    for short in ['07', '90', '120']:
        global_rate = gr[short_to_target[short]]
        df[f'county_vs_global_{short}'] = df[f'county_{short}_rate'] - global_rate
        df[f'topic_vs_global_{short}']  = df[f'topic_avg_{short}_rate'] - global_rate

    # ---- Label encoding haute cardinalité ----
    for col in ['county', 'subcounty', 'ward', 'group_name', 'trainer_first']:
        le = LabelEncoder()
        df[f'{col}_le'] = le.fit_transform(df[col].fillna('unknown').astype(str))

    # ---- Nettoyage colonnes temporaires ----
    drop_cols = [f'{c1}_{c2}' for (c1, c2) in [
        ('county', 'gender'), ('county', 'age'), ('ward', 'belong_to_cooperative'),
        ('group_name', 'gender'), ('county', 'registration')
    ] if f'{c1}_{c2}' in df.columns]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    return df


def get_feature_cols(train_fe):
    exclude = [
        'ID', 'farmer_name', 'training_day', 'gender', 'age', 'registration',
        'group_name', 'county', 'subcounty', 'ward', 'trainer', 'topics_list',
        'trainer_first', 'topics_parsed', 'trainers_parsed',
        'adopted_within_07_days', 'adopted_within_90_days', 'adopted_within_120_days',
    ]
    cols = [c for c in train_fe.columns
            if c not in exclude
            and train_fe[c].dtype in [np.float64, np.int64, np.float32, np.int32]
            and not train_fe[c].isnull().all()]
    return cols


# ============================================================
# 4. MODÈLES
# ============================================================

def get_models(target):
    pos_weights = {
        'adopted_within_07_days': 80,
        'adopted_within_90_days': 60,
        'adopted_within_120_days': 40,
    }
    pw = pos_weights[target]
    class_weight = {0: 1, 1: pw}

    models = {}

    # HistGBM (toujours disponible)
    models['hgb1'] = HistGradientBoostingClassifier(
        max_iter=800, learning_rate=0.03, max_depth=6,
        min_samples_leaf=15, l2_regularization=0.05,
        class_weight='balanced', early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=40, random_state=42,
    )
    models['hgb2'] = HistGradientBoostingClassifier(
        max_iter=800, learning_rate=0.05, max_depth=8,
        min_samples_leaf=20, l2_regularization=0.2,
        class_weight='balanced', early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=40, random_state=123,
    )

    models['rf'] = RandomForestClassifier(
        n_estimators=600, max_depth=12, min_samples_leaf=8,
        class_weight=class_weight, n_jobs=-1, random_state=42,
        max_features='sqrt',
    )

    if USE_LGBM:
        # Deux configs LightGBM différentes pour diversité
        models['lgbm1'] = lgb.LGBMClassifier(
            n_estimators=3000, learning_rate=0.02, num_leaves=63,
            max_depth=7, min_child_samples=20, subsample=0.8,
            colsample_bytree=0.8, scale_pos_weight=pw,
            reg_alpha=0.05, reg_lambda=1.0,
            random_state=42, n_jobs=-1, verbose=-1,
        )
        models['lgbm2'] = lgb.LGBMClassifier(
            n_estimators=3000, learning_rate=0.02, num_leaves=31,
            max_depth=5, min_child_samples=30, subsample=0.7,
            colsample_bytree=0.7, scale_pos_weight=pw,
            reg_alpha=0.1, reg_lambda=2.0,
            random_state=99, n_jobs=-1, verbose=-1,
        )

    if USE_XGB:
        models['xgb'] = xgb.XGBClassifier(
            n_estimators=3000, learning_rate=0.02, max_depth=6,
            min_child_weight=8, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pw, random_state=42, n_jobs=-1,
            verbosity=0, early_stopping_rounds=100, eval_metric='logloss',
            reg_alpha=0.05, reg_lambda=1.0,
        )

    if USE_CATBOOST:
        models['catboost'] = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=7,
            scale_pos_weight=pw, random_seed=42, verbose=0,
            early_stopping_rounds=100, l2_leaf_reg=3.0,
        )

    return models


# ============================================================
# 5. CROSS-VALIDATION
# ============================================================

def train_cv(train_fe, test_fe, feature_cols, target, n_splits=N_SPLITS):
    X      = train_fe[feature_cols].fillna(-1).values
    y      = train_fe[target].values
    X_test = test_fe[feature_cols].fillna(-1).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    models_dict = get_models(target)

    oof_preds  = {name: np.zeros(len(X))      for name in models_dict}
    test_preds = {name: np.zeros(len(X_test)) for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"    Fold {fold+1}/{n_splits}...", end=' ')
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        for name, model in models_dict.items():
            if USE_LGBM and name.startswith('lgbm'):
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(150, verbose=False),
                                     lgb.log_evaluation(-1)])
            elif USE_XGB and name == 'xgb':
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            elif USE_CATBOOST and name == 'catboost':
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
            else:
                model.fit(X_tr, y_tr)

            oof_preds[name][val_idx] = model.predict_proba(X_val)[:, 1]
            test_preds[name] += model.predict_proba(X_test)[:, 1] / n_splits

        print("OK")

    return oof_preds, test_preds


# ============================================================
# 6. PSEUDO-LABELING
# ============================================================

def pseudo_label_cv(train_fe, test_fe, feature_cols, target,
                    test_preds_init, n_splits=N_SPLITS):
    """
    Ajoute les exemples test très confiants au train, réentraîne.
    """
    avg_test_pred = np.mean(list(test_preds_init.values()), axis=0)

    hi_mask = avg_test_pred >= PSEUDO_THR_HI
    lo_mask = avg_test_pred <= PSEUDO_THR_LO

    n_hi = hi_mask.sum()
    n_lo = lo_mask.sum()
    print(f"  Pseudo-labeling: {n_hi} positifs, {n_lo} négatifs")

    if n_hi + n_lo == 0:
        return test_preds_init

    pseudo_X = test_fe[feature_cols].fillna(-1).values
    pseudo_y_hi = np.ones(n_hi)
    pseudo_y_lo = np.zeros(n_lo)

    X_orig = train_fe[feature_cols].fillna(-1).values
    y_orig = train_fe[target].values

    X_aug = np.vstack([X_orig,
                       pseudo_X[hi_mask],
                       pseudo_X[lo_mask]])
    y_aug = np.concatenate([y_orig, pseudo_y_hi, pseudo_y_lo])

    X_test = pseudo_X

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=77)
    models_dict = get_models(target)
    test_preds_pseudo = {name: np.zeros(len(X_test)) for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_aug, y_aug)):
        # On ne valide que sur les vrais exemples originaux
        val_idx_orig = val_idx[val_idx < len(X_orig)]
        if len(val_idx_orig) == 0:
            continue
        X_tr, X_val = X_aug[tr_idx], X_aug[val_idx_orig]
        y_tr, y_val = y_aug[tr_idx], y_aug[val_idx_orig]

        for name, model in models_dict.items():
            if USE_LGBM and name.startswith('lgbm'):
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(150, verbose=False),
                                     lgb.log_evaluation(-1)])
            elif USE_XGB and name == 'xgb':
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            elif USE_CATBOOST and name == 'catboost':
                model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
            else:
                model.fit(X_tr, y_tr)

            test_preds_pseudo[name] += model.predict_proba(X_test)[:, 1] / n_splits

    return test_preds_pseudo


# ============================================================
# 7. STACKING NIVEAU 2
# ============================================================

def stack(oof_preds, test_preds, y):
    """
    Stacking avec XGBoost (ou LogReg si non dispo) + calibration isotonique.
    """
    model_names = list(oof_preds.keys())
    X_meta_train = np.column_stack([oof_preds[n] for n in model_names])
    X_meta_test  = np.column_stack([test_preds[n] for n in model_names])

    # Moyenne simple (robuste)
    avg_oof  = np.mean(X_meta_train, axis=1)
    avg_test = np.mean(X_meta_test, axis=1)

    if USE_XGB and len(model_names) > 1:
        meta = xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=2,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(1 - y.mean()) / (y.mean() + 1e-6),
            random_state=42, verbosity=0,
        )
        # CV pour le meta-learner
        skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        meta_oof  = np.zeros(len(y))
        meta_test = np.zeros(len(X_meta_test))
        for tr_idx, val_idx in skf5.split(X_meta_train, y):
            meta.fit(X_meta_train[tr_idx], y[tr_idx])
            meta_oof[val_idx] = meta.predict_proba(X_meta_train[val_idx])[:, 1]
            meta_test += meta.predict_proba(X_meta_test)[:, 1] / 5
    elif len(model_names) > 1:
        meta = LogisticRegression(C=0.05, max_iter=1000, random_state=42)
        skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        meta_oof  = np.zeros(len(y))
        meta_test = np.zeros(len(X_meta_test))
        for tr_idx, val_idx in skf5.split(X_meta_train, y):
            meta.fit(X_meta_train[tr_idx], y[tr_idx])
            meta_oof[val_idx] = meta.predict_proba(X_meta_train[val_idx])[:, 1]
            meta_test += meta.predict_proba(X_meta_test)[:, 1] / 5
    else:
        meta_oof  = avg_oof
        meta_test = avg_test

    # Blend meta + moyenne simple pour robustesse
    final_oof  = 0.7 * meta_oof  + 0.3 * avg_oof
    final_test = 0.7 * meta_test + 0.3 * avg_test

    # Calibration isotonique sur OOF
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(final_oof, y)
    final_oof_cal  = iso.predict(final_oof)
    final_test_cal = iso.predict(final_test)

    return final_oof_cal, final_test_cal


# ============================================================
# 8. ÉVALUATION
# ============================================================

def evaluate(y_true, y_pred, target_name):
    ll  = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)

    thresholds = np.linspace(0.01, 0.5, 200)
    f1s = [f1_score(y_true, (y_pred >= t).astype(int), zero_division=0) for t in thresholds]
    best_f1  = max(f1s)
    best_thr = thresholds[np.argmax(f1s)]

    challenge_score = 0.75 * (1 - ll) + 0.25 * auc

    print(f"\n  ✓ [{target_name}]")
    print(f"    LogLoss:   {ll:.5f}")
    print(f"    ROC-AUC:   {auc:.5f}")
    print(f"    F1 (best): {best_f1:.5f} @ threshold={best_thr:.3f}")
    print(f"    Score≈:    {challenge_score:.5f}")
    return ll, auc, best_f1


# ============================================================
# 9. PIPELINE PRINCIPAL
# ============================================================

def main(data_path='./'):
    print("="*65)
    print("  DigiCow Adoption Prediction — Solution v2 (améliorée)")
    print("="*65)

    # 1. Chargement
    print("\n[1] Chargement des données...")
    train  = pd.read_csv(f'{data_path}Train.csv')
    test   = pd.read_csv(f'{data_path}Test.csv')
    prior  = pd.read_csv(f'{data_path}Prior.csv')
    sample = pd.read_csv(f'{data_path}SampleSubmission.csv')
    print(f"  Train: {train.shape} | Test: {test.shape} | Prior: {prior.shape}")
    print(f"  Taux adoption 7j:   {train['adopted_within_07_days'].mean():.3%}")
    print(f"  Taux adoption 90j:  {train['adopted_within_90_days'].mean():.3%}")
    print(f"  Taux adoption 120j: {train['adopted_within_120_days'].mean():.3%}")

    # 2. Stats Prior
    print("\n[2] Construction des statistiques Prior...")
    prior_stats = build_prior_stats(prior)

    # 3. Feature Engineering
    print("\n[3] Feature Engineering...")
    train_fe = feature_engineer(train, prior_stats, is_train=True)
    test_fe  = feature_engineer(test,  prior_stats, is_train=False)
    feature_cols = get_feature_cols(train_fe)
    print(f"  Nombre de features: {len(feature_cols)}")

    # 4. Entraînement
    submission = sample.copy()
    results = {}

    TARGET_MAP = [
        ('adopted_within_07_days',  'Target_07_AUC',  'Target_07_LogLoss'),
        ('adopted_within_90_days',  'Target_90_AUC',  'Target_90_LogLoss'),
        ('adopted_within_120_days', 'Target_120_AUC', 'Target_120_LogLoss'),
    ]

    for target, col_auc, col_ll in TARGET_MAP:
        print(f"\n[4] ══════ Entraînement: {target} ══════")
        y = train_fe[target].values

        # Cross-validation initiale
        oof_preds, test_preds_init = train_cv(
            train_fe, test_fe, feature_cols, target, n_splits=N_SPLITS
        )

        # Pseudo-labeling (optionnel)
        if USE_PSEUDO:
            print("  Pseudo-labeling...")
            test_preds_pseudo = pseudo_label_cv(
                train_fe, test_fe, feature_cols, target,
                test_preds_init, n_splits=N_SPLITS
            )
            # Blend: 70% original + 30% pseudo
            test_preds_blended = {}
            for name in test_preds_init:
                test_preds_blended[name] = (
                    0.7 * test_preds_init[name] + 0.3 * test_preds_pseudo[name]
                )
        else:
            test_preds_blended = test_preds_init

        # Stacking
        print("  Stacking niveau 2 + calibration isotonique...")
        final_oof, final_test = stack(oof_preds, test_preds_blended, y)

        # Évaluation OOF
        ll, auc, f1 = evaluate(y, final_oof, target)
        results[target] = {'logloss': ll, 'auc': auc, 'f1': f1}

        submission[col_auc] = final_test
        submission[col_ll]  = final_test

    # 5. Résumé
    print("\n" + "="*65)
    print("  RÉSUMÉ FINAL (OOF Cross-Validation)")
    print("="*65)
    total_score = 0
    for target, metrics in results.items():
        score = 0.75 * (1 - metrics['logloss']) + 0.25 * metrics['auc']
        total_score += score / 3
        print(f"  {target[-3:]}j → LogLoss={metrics['logloss']:.4f} | "
              f"AUC={metrics['auc']:.4f} | F1={metrics['f1']:.4f} | Score≈{score:.4f}")
    print(f"\n  Score global estimé: {total_score:.5f}")

    # 6. Sauvegarde
    print("\n[5] Sauvegarde...")
    submission.to_csv('submission_v2.csv', index=False)
    print("  ✓ submission_v2.csv créé")

    # Distribution des prédictions
    print("\n  Distribution prédictions test:")
    for short, col in [('7j', 'Target_07_AUC'), ('90j', 'Target_90_AUC'), ('120j', 'Target_120_AUC')]:
        p = submission[col]
        print(f"    [{short}] Min={p.min():.4f} Max={p.max():.4f} "
              f"Mean={p.mean():.4f} Median={p.median():.4f}")

    return submission



if __name__ == '__main__':
    # Modifiez ce chemin si vos CSV sont dans un autre dossier
    data_path = './'
    sub = main(data_path=data_path)
    print("\nTerminé ! Soumettez submission_v2.csv")   '( 

✓ LightGBM disponible
✓ XGBoost disponible
✓ CatBoost disponible
  DigiCow Adoption Prediction — Solution v2 (améliorée)

[1] Chargement des données...
  Train: (13536, 17) | Test: (5621, 14) | Prior: (44882, 17)
  Taux adoption 7j:   1.130%
  Taux adoption 90j:  1.581%
  Taux adoption 120j: 2.231%

[2] Construction des statistiques Prior...
  Calcul des stats Prior...

[3] Feature Engineering...
  Nombre de features: 193

[4] ══════ Entraînement: adopted_within_07_days ══════
    Fold 1/10... OK
    Fold 2/10... OK
    Fold 3/10... OK
    Fold 4/10... OK
    Fold 5/10... OK
    Fold 6/10... OK
    Fold 7/10... OK
    Fold 8/10... OK
    Fold 9/10... OK
    Fold 10/10... OK
  Pseudo-labeling...
  Pseudo-labeling: 0 positifs, 4437 négatifs
  Stacking niveau 2 + calibration isotonique...

  ✓ [adopted_within_07_days]
    LogLoss:   0.02135
    ROC-AUC:   0.98988
    F1 (best): 0.59259 @ threshold=0.281
    Score≈:    0.98146

[4] ══════ Entraînement: adopted_within_90_days ══════
    Fol